In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
    importlib.reload(sys.modules['helpers.database'])
    importlib.reload(sys.modules['helpers.logging_config'])

In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================

from helpers import create_connection, load_bronze_table, write_gold_table, setup_logger
from pyspark.sql.functions import xxhash64, col, datediff, when
import pyspark.sql.functions as F
import time

# Setup logging
logger = setup_logger("load_facts")

# Initialize connection
c = create_connection(spark, dbutils)
logger.info("=" * 70)
logger.info("FACT LOAD JOB STARTED")
logger.info("=" * 70)


In [0]:
# ==============================================================================
# BRONZE LAYER: Load source tables (in-memory DataFrames)
# ==============================================================================

service_bronze = load_bronze_table(c, "service")
rental_bronze = load_bronze_table(c, "rental")
staff_bronze = load_bronze_table(c, "staff")
inventory_bronze = load_bronze_table(c, "inventory")
payment_bronze = load_bronze_table(c, "payment")


In [0]:
# ==============================================================================
# SILVER LAYER: Clean and prepare DataFrames
# ==============================================================================

logger.info("SILVER: Preparing cleaned DataFrames")

# For now, no transformations needed (source data is clean)
# If data quality issues found, add corrections here
service_silver = service_bronze
rental_silver = rental_bronze
staff_silver = staff_bronze
inventory_silver = inventory_bronze
payment_silver = payment_bronze

logger.info("SILVER: Data preparation complete")


In [0]:
# ==============================================================================
# GOLD: FACT_SERVICE
# ==============================================================================

logger.info("GOLD: Building fact_service")
start_time = time.time()

fact_service = service_silver.select(
    "service_id",
    "service_date",
    "service_type",
    "service_cost",
    "inventory_id",
).withColumn(
    "service_key", xxhash64(col("service_id"))
).withColumn(
    "service_date_key", xxhash64(col("service_date"))  # References dim_service_date
).withColumn(
    "car_key", xxhash64(col("inventory_id"))
).drop(
    "service_date",
    "inventory_id"
)

write_gold_table(fact_service, "fact_service", mode="overwrite")
logger.info(f"GOLD: fact_service completed in {time.time() - start_time:.2f}s")


In [ ]:
# ==============================================================================
# GOLD: FACT_RENTAL
# ==============================================================================

logger.info("GOLD: Building fact_rental")
start_time = time.time()

# Prepare staff DataFrame with aliases
staff_prep = staff_silver.select(
    col("staff_id").alias("staff_staff_id"),
    "store_id"
)

# Prepare payment DataFrame with aliases
payment_prep = payment_silver.select(
    col("rental_id").alias("payment_rental_id"),
    "payment_date",
    col("amount").alias("payment_amount")
)

# Join rental with staff
rental_with_staff = rental_silver.join(
    staff_prep,
    rental_silver.staff_id == staff_prep.staff_staff_id,
    "left"
)

# Join with inventory to get car_id
rental_with_staff_inventory = rental_with_staff.join(
    inventory_silver.select("inventory_id", "car_id"),
    rental_with_staff.inventory_id == inventory_silver.inventory_id,
    "left"
)

# Join with payment
rental_with_staff_inventory_payment = rental_with_staff_inventory.join(
    payment_prep,
    rental_with_staff_inventory.rental_id == payment_prep.payment_rental_id,
    "left"
)

# Build fact table with surrogate keys and calculated columns
fact_rental = (
    rental_with_staff_inventory_payment.select(
        "rental_id",
        "rental_rate",
        "payment_amount",
        "customer_id",
        "car_id",
        "staff_staff_id",
        "store_id",
        "rental_date",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
    .withColumn("rental_key", xxhash64(col("rental_id")))
    .withColumn("customer_key", xxhash64(col("customer_id")))
    .withColumn("car_key", xxhash64(col("car_id")))
    .withColumn("staff_key", xxhash64(col("staff_staff_id")))
    .withColumn("store_key", xxhash64(col("store_id")))
    .withColumn("rental_date_key", xxhash64(col("rental_date")))
    .withColumn("return_date_key",
        F.when(col("return_date").isNotNull(), xxhash64(col("return_date"))).otherwise(None))
    .withColumn("payment_date_key",
        F.when(col("payment_date").isNotNull(), xxhash64(col("payment_date"))).otherwise(None))
    .withColumn("payment_deadline_date_key", xxhash64(col("payment_deadline")))
    .withColumn(
        "rental_amount",
        col("rental_rate") * datediff(col("return_date"), col("rental_date"))
    )
    .withColumn("rental_duration", datediff(col("return_date"), col("rental_date")))
    .withColumn(
        "payment_delay_duration",
        datediff(col("payment_date"), col("payment_deadline"))
    )
    .drop(
        "customer_id",
        "car_id",
        "staff_staff_id",
        "store_id",
        "rental_date",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
)

write_gold_table(fact_rental, "fact_rental", mode="overwrite")
logger.info(f"GOLD: fact_rental completed in {time.time() - start_time:.2f}s")

# ==============================================================================
# JOB COMPLETION
# ==============================================================================
logger.info("=" * 70)
logger.info("FACT LOAD JOB COMPLETED SUCCESSFULLY")
logger.info("=" * 70)
